# CSE 438

## Group Members

| Name | Student ID |
|------|------------|
| Sumiaya Sharif | 2023-1-60-048 |
| Ohiduzzaman Rayhan | 2023-1-60-108 |
| Prottoy Modak | 2022-3-60-130 |
| Afrina Alam Lamia | 2022-1-60-231 |

---

## Setup & imports (version pins)

In [ ]:
!pip uninstall -y numpy scikit-image albumentations albucore opencv-python opencv-python-headless
!pip install -q --no-cache-dir numpy==1.26.4
!pip install -q --no-cache-dir scikit-image==0.24.0
!pip install -q --no-cache-dir opencv-python-headless==4.10.0.84
!pip install -q --no-cache-dir albumentations==1.4.24

In [ ]:
import numpy as np
import skimage
import albumentations as A
import cv2

from skimage.measure import label
from skimage.transform import ProjectiveTransform

print("NumPy:", np.__version__)
print("scikit-image:", skimage.__version__)
print("Albumentations:", A.__version__)
print("OpenCV:", cv2.__version__)

print("\n✅ All image-processing imports are working!")

In [ ]:
!pip install -q --force-reinstall --no-deps "scikit-image==0.22.0"
!pip install -q "albumentations==1.4.15" "albucore==0.0.16"
!pip install -q "transformers==4.44.2" "opencv-python-headless==4.10.0.84" "timm==1.0.9"

print("Installation complete.")
print("IMPORTANT: Restart the Kaggle session before running the remaining cells.")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import os, json, time, random, math, copy
import numpy as np
import pandas as pd
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt

from transformers import (
    ViTMAEModel,
    ViTMAEForPreTraining,
    ViTMAEConfig
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Torch:", torch.__version__)
print("Transformers imported successfully")
print("Albumentations imported successfully")

In [ ]:
import skimage
import scipy
import albumentations
from skimage.transform import ProjectiveTransform

print("scikit-image:", skimage.__version__)
print("scipy:", scipy.__version__)
print("albumentations:", albumentations.__version__)
print("ProjectiveTransform: OK")

## Master config cell

In [ ]:
CONFIG = {
    "nb0_dataset_slug": "/kaggle/input/datasets/shaheb366/delta-food/nb0_artifacts",       # Kaggle dataset with nb0_artifacts/
    "dataprep_dir_slug": "/kaggle/input/datasets/afrinaalam/partb-outputs",     # Kaggle dataset with partb_dataprep_artifacts/
    "raw_dataset_slug": "/kaggle/input/datasets/shaheb366/raw-dataset",         # try this first; else your own uploaded copy
    # -------------------------------------------------------------

    "out_dir": "/kaggle/working/mae_outputs",

    "hf_backbone": "facebook/vit-mae-base",   # ViT-Base/16 MAE, ImageNet-pretrained -> we CONTINUE SSL pretraining on FoodBD
    "patch_size": 16,
    "encoder_img_size": 224,                  # 14*16 -- native ViT-B/16 MAE resolution
    "mask_ratio": 0.75,                       # ~75% of patches masked; encoder only ever sees the visible 25%
    "layer_taps": [2, 5, 8, 11],               # 4 of 12 ViT-Base encoder depths tapped for the Task C adapter

    "mae_epochs": 50,
    "mae_batch_size": 24,
    "mae_base_lr": 1.5e-4,
    "mae_min_lr": 1e-6,
    "mae_warmup_epochs": 5,
    "mae_weight_decay": 0.05,
    "norm_pix_loss": True,                    # reconstruct *normalized* pixel targets (MAE paper default, improves quality)

    "decoder_embed_dim": 256,                 # width of the DOWNSTREAM SegFormer-style seg head (Task C) -- NOT the MAE pixel-decoder
    "ft_epochs": 50,
    "ft_batch_size": 8,
    "ft_lr_encoder": 1e-5,
    "ft_lr_decoder": 1e-4,
    "ft_weight_decay": 1e-4,
    "encoder_mode": "finetune",

    "seed": SEED,
}
os.makedirs(CONFIG["out_dir"], exist_ok=True)
print(json.dumps(CONFIG, indent=2, default=str))

## Load Task‑A split roles

In [ ]:
import sys

def _resolve(slug):
    return slug if slug.startswith("/") else f"/kaggle/input/{slug}"

# --- NB0 artifacts (your real seg_common.py) ---
NB0_DIR = _resolve(CONFIG["nb0_dataset_slug"])
assert os.path.exists(f"{NB0_DIR}/common_config.json"), (
    f"common_config.json not found under {NB0_DIR}. "
    f"Run `!ls -R /kaggle/input` and fix CONFIG['nb0_dataset_slug'].")
sys.path.insert(0, NB0_DIR)
from seg_common import load_common_config, load_split_manifest, polygons_to_mask, get_transforms, FoodSegDataset

cfg = load_common_config(f"{NB0_DIR}/common_config.json")
IMG_SIZE_PARTA = cfg["img_size"]                        # 512, Part-A's resolution (reference only)
NUM_CLASSES = cfg["num_classes_total"]                  # 68 (67 food classes + background)
CLASS_NAMES = ["background"] + cfg["class_names"]
CONFIG["num_classes"] = NUM_CLASSES
print(f"NUM_CLASSES={NUM_CLASSES} | Part-A IMG_SIZE={IMG_SIZE_PARTA} | MAE encoder IMG_SIZE={CONFIG['encoder_img_size']}")

# --- Part-B role-mapped split (Task A, already done) ---
DATAPREP_DIR = _resolve(CONFIG["dataprep_dir_slug"])
sys.path.insert(0, DATAPREP_DIR)
from ssl_common import load_partb_config, load_remapped_split

partb_cfg = load_partb_config(f"{DATAPREP_DIR}/partb_config.json")
splits = load_remapped_split(f"{DATAPREP_DIR}/split_manifest_remapped.json")

# --- point paths at whichever raw-dataset copy you actually attached ---
BAKED_ROOT = cfg["data_root"]                            # what NB0 recorded when it ran
RAW_ROOT = f"{_resolve(CONFIG['raw_dataset_slug'])}/FoodBD A Polygon-Annotated Meal Image Dataset of B/root/FoodBD"

def _remap(p):
    return p.replace(BAKED_ROOT, RAW_ROOT) if isinstance(p, str) else p

for key in ["train", "valid", "test"]:
    splits[key] = [_remap(p) for p in splits[key]]

train_imgs = splits["train"]   # 2473 -> unlabelled SSL pretraining corpus (Task B)
valid_imgs = splits["valid"]   # 697  -> labelled fine-tuning set (Task D)
test_imgs  = splits["test"]    # 353  -> downstream monitoring + final eval (Task D & E, double role)

_missing = [p for p in (train_imgs[:3] + valid_imgs[:3] + test_imgs[:3]) if not os.path.exists(p)]
if _missing:
    print("RAW DATASET NOT FOUND at:")
    for p in _missing: print("  ", p)
    print("\nMounted under /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        if files: print("  ", root, "->", len(files), "files, e.g.", files[0])
    raise FileNotFoundError("Attach the raw FoodBD dataset, or fix CONFIG['raw_dataset_slug'].")
print(f"Paths resolve OK. train={len(train_imgs)} valid={len(valid_imgs)} test={len(test_imgs)}")

# sanity: a label file should exist next to the first train image
sample_label = train_imgs[0].replace("/images/", "/labels/").rsplit(".",1)[0] + ".txt"
print("Sample label file exists:", os.path.exists(sample_label), "->", sample_label)

## Transforms (reuse Part‑A's get_transforms, now at 224px for the ViT‑B/16 MAE encoder)

In [ ]:
# MAE pretraining augmentation (Task B, images only) -- official MAE recipe: RandomResizedCrop + HFlip ONLY.
# (No strong color jitter here, unlike the DINOv2 notebook: MAE's objective is pixel reconstruction, so
# heavy photometric distortion would corrupt the very target the decoder is trying to predict.)
mae_tf = A.Compose([
    A.RandomResizedCrop(size=(CONFIG["encoder_img_size"], CONFIG["encoder_img_size"]),
                         scale=(0.2, 1.0), ratio=(0.75, 1.33)),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

# Task D/E: identical synced image+mask pipeline as Part-A, resized to the encoder's patch-16-compatible
# resolution (224) instead of Part-A's 512 -- this is the Task C adapter.
IMG_SIZE = CONFIG["encoder_img_size"]
finetune_tf = get_transforms(IMG_SIZE, train=True)    # labelled fine-tune (valid split)
eval_tf     = get_transforms(IMG_SIZE, train=False)   # deterministic (test split)

## Datasets (FoodSegDataset for labelled data, custom loader for unlabelled)

In [ ]:
class MAEDataset(Dataset):
    """Task B: images-only, no masks. Unlike DINOv2's multi-crop teacher/student setup, MAE needs just
    ONE augmented view per image -- the ~75% patch masking that drives the SSL signal happens *inside*
    the model itself (random per forward pass), not in the dataloader."""
    def __init__(self, img_paths, tf):
        self.img_paths = img_paths
        self.tf = tf

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = cv2.cvtColor(cv2.imread(str(self.img_paths[idx])), cv2.COLOR_BGR2RGB)
        return self.tf(image=img)["image"]

mae_ds = MAEDataset(train_imgs, mae_tf)

# Task D & E: reuse Part-A's exact FoodSegDataset (polygon->mask logic), same bg_is_zero flag
ft_ds   = FoodSegDataset(valid_imgs, finetune_tf, bg_is_zero=cfg["bg_is_zero"])
test_ds = FoodSegDataset(test_imgs,  eval_tf,     bg_is_zero=cfg["bg_is_zero"])

mae_loader  = DataLoader(mae_ds, batch_size=CONFIG["mae_batch_size"], shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
ft_loader   = DataLoader(ft_ds, batch_size=CONFIG["ft_batch_size"], shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=CONFIG["ft_batch_size"], shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"MAE batches/epoch: {len(mae_loader)} | FT batches/epoch: {len(ft_loader)} | Test batches: {len(test_loader)}")

## Sanity checks

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

In [ ]:
def denorm(t):
    m = np.array(IMAGENET_MEAN).reshape(3,1,1); s = np.array(IMAGENET_STD).reshape(3,1,1)
    x = t.numpy() * s + m
    return np.clip(x.transpose(1,2,0), 0, 1)

# (a) one augmented MAE input view of an unlabelled train image (pre-masking)
sample_view = mae_ds[0]
fig, ax = plt.subplots(1,1, figsize=(4,4))
ax.imshow(denorm(sample_view)); ax.set_title("MAE input view (pre-masking)"); ax.axis("off")
plt.tight_layout(); plt.show()

# (b) validation-split image with mask overlay -- CHECK THIS LOOKS RIGHT before trusting derive_mask_path()
img_t, mask_t = ft_ds[0]
img_np = denorm(img_t)
fig, ax = plt.subplots(1,1, figsize=(5,5))
ax.imshow(img_np)
ax.imshow(mask_t.numpy(), alpha=0.45, cmap="tab20")
ax.set_title(f"val image + mask overlay | unique classes: {torch.unique(mask_t).tolist()[:10]}...")
ax.axis("off")
plt.show()

## MAE backbone (ViTMAEForPreTraining) + Task‑C adapter (token→2D‑grid reshape)

In [ ]:
def build_mae_model(name=CONFIG["hf_backbone"], mask_ratio=CONFIG["mask_ratio"], norm_pix_loss=CONFIG["norm_pix_loss"]):
    """Loads an ImageNet-pretrained ViT-Base/16 MAE checkpoint (encoder + lightweight decoder + mask
    token), then we CONTINUE MAE self-supervised pretraining (masked-patch reconstruction) on the
    unlabelled FoodBD corpus -- exactly mirrors how the DINOv2 notebook starts from an already
    ImageNet-pretrained backbone rather than random init."""
    config = ViTMAEConfig.from_pretrained(name)
    config.mask_ratio = mask_ratio
    config.norm_pix_loss = norm_pix_loss
    model = ViTMAEForPreTraining.from_pretrained(name, config=config)
    return model


def get_multilevel_tokens(vit_encoder, pixel_values, layer_idxs, patch_size):
    """
    Task C adapter: same idea as the DINOv2 notebook -- ViT-Base/16 is a *non-hierarchical* ViT, so we
    tap 4 different transformer DEPTHS (not 4 spatial scales) to feed a 4-input SegFormer-style
    All-MLP head. Crucially, for DOWNSTREAM segmentation the encoder must see the *whole* image:
    random masking is a pretraining-only trick that forces the encoder to learn from partial context;
    at fine-tuning/inference time the standard MAE protocol feeds every patch back in.

    We force mask_ratio=0 AND pass identity noise (0,1,2,...) so the internal random-shuffle step
    keeps every patch (since none are dropped) in its *original raster order* -- without the
    identity noise, HF's random_masking would still shuffle token order even at mask_ratio=0,
    which would silently scramble the (Hp,Wp) grid reshape below.
    """
    B, C, H, W = pixel_values.shape
    Hp, Wp = H // patch_size, W // patch_size
    seq_len = Hp * Wp

    vit = vit_encoder
    orig_ratio = vit.config.mask_ratio
    vit.config.mask_ratio = 0.0
    identity_noise = torch.arange(seq_len, device=pixel_values.device).float().unsqueeze(0).expand(B, -1)
    out = vit(pixel_values=pixel_values, noise=identity_noise, output_hidden_states=True)
    vit.config.mask_ratio = orig_ratio

    hidden_states = out.hidden_states  # tuple: [embeddings, layer1, layer2, ..., layerN]
    maps = []
    for idx in layer_idxs:
        tokens = hidden_states[idx + 1]        # +1 to skip the embedding-output entry
        tokens = tokens[:, 1:, :]               # drop CLS token, keep patch tokens
        fmap = tokens.transpose(1, 2).reshape(B, -1, Hp, Wp)  # (B, C, Hp, Wp) adapter reshape
        maps.append(fmap)
    return maps  # list of 4 tensors, each (B, embed_dim_backbone, Hp, Wp)

## Masked-patch reconstruction visualization utility

In [ ]:
def visualize_mae_reconstruction(model, img_batch, n_show=4, tag=""):
    """Reproduces the classic MAE qualitative figure: original | masked input | reconstruction
    (visible patches copied through, masked patches filled in by the decoder's prediction)."""
    model.eval()
    with torch.no_grad():
        outputs = model(pixel_values=img_batch.to(DEVICE))
    patch_size = model.config.patch_size
    recon = model.unpatchify(outputs.logits)                          # (B,3,H,W) predicted pixels
    mask = outputs.mask.unsqueeze(-1).repeat(1, 1, patch_size**2 * 3)
    mask_img = model.unpatchify(mask)                                  # 1 = masked patch, 0 = visible patch

    imgs = img_batch.to(DEVICE)
    masked_input = imgs * (1 - mask_img)                               # visible patches only
    pasted_recon = imgs * (1 - mask_img) + recon * mask_img            # keep visible pixels, paste recon on masked ones

    fig, axes = plt.subplots(n_show, 3, figsize=(9, 3*n_show))
    for i in range(n_show):
        axes[i,0].imshow(denorm(imgs[i].cpu())); axes[i,0].set_title("original"); axes[i,0].axis("off")
        axes[i,1].imshow(denorm(masked_input[i].cpu())); axes[i,1].set_title(f"~{int(CONFIG['mask_ratio']*100)}% masked"); axes[i,1].axis("off")
        axes[i,2].imshow(denorm(pasted_recon[i].cpu())); axes[i,2].set_title("reconstruction"); axes[i,2].axis("off")
    plt.tight_layout(); plt.savefig(f"{CONFIG['out_dir']}/mae_reconstructions_{tag}.png", dpi=150); plt.show()

## Build MAE model, schedules, optimizer

In [ ]:
def cosine_schedule(base, final, epochs, warmup_epochs=0, warmup_start=0.0):
    warmup = np.linspace(warmup_start, base, warmup_epochs)
    remain = epochs - warmup_epochs
    t = np.arange(remain)
    cosine = final + 0.5 * (base - final) * (1 + np.cos(np.pi * t / remain))
    return np.concatenate([warmup, cosine])

EP = CONFIG["mae_epochs"]
lr_sched = cosine_schedule(CONFIG["mae_base_lr"] * CONFIG["mae_batch_size"] / 256,
                            CONFIG["mae_min_lr"], EP, CONFIG["mae_warmup_epochs"])

mae_model = build_mae_model().to(DEVICE)
embed_dim = mae_model.config.hidden_size

optimizer = torch.optim.AdamW(mae_model.parameters(), lr=lr_sched[0], weight_decay=CONFIG["mae_weight_decay"])
scaler = torch.cuda.amp.GradScaler()

print(f"Backbone: {CONFIG['hf_backbone']} | embed_dim={embed_dim} | mask_ratio={CONFIG['mask_ratio']} | "
      f"params(MAE incl. decoder)={sum(p.numel() for p in mae_model.parameters())/1e6:.1f}M")

# quick baseline check: how does the OFF-THE-SHELF ImageNet-MAE reconstruct FoodBD images,
# before any continued pretraining on this domain?
sample_batch = torch.stack([mae_ds[i] for i in range(4)])
visualize_mae_reconstruction(mae_model, sample_batch, n_show=4, tag="before_ssl")

## Task B: MAE pretraining loop (50 epochs, timed, checkpointed)

In [ ]:
os.makedirs(f"{CONFIG['out_dir']}/mae_ckpts", exist_ok=True)
mae_loss_history, mae_epoch_times = [], []

for epoch in range(CONFIG["mae_epochs"]):
    t0 = time.time()
    for g in optimizer.param_groups:
        g["lr"] = lr_sched[epoch]

    mae_model.train()
    running_loss, n_batches = 0.0, 0
    for imgs in mae_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast():
            outputs = mae_model(pixel_values=imgs)   # random ~75% patch masking happens inside the model
            loss = outputs.loss                       # MSE on masked patches only (normalized pixels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(mae_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item(); n_batches += 1

    epoch_loss = running_loss / n_batches
    epoch_time = time.time() - t0
    mae_loss_history.append(epoch_loss)
    mae_epoch_times.append(epoch_time)
    print(f"[MAE] epoch {epoch+1}/{CONFIG['mae_epochs']} | recon_loss={epoch_loss:.4f} | {epoch_time:.1f}s/epoch")

    if (epoch + 1) % 10 == 0 or epoch == CONFIG["mae_epochs"] - 1:
        torch.save(mae_model.state_dict(), f"{CONFIG['out_dir']}/mae_ckpts/mae_model_ep{epoch+1}.pth")

print(f"Avg per-epoch MAE time: {np.mean(mae_epoch_times):.1f}s "
      f"({'>10min -> consider splitting into pretrain/downstream notebooks' if np.mean(mae_epoch_times) > 600 else 'OK within a single notebook'})")

plt.figure(figsize=(6,4))
plt.plot(mae_loss_history)
plt.xlabel("epoch"); plt.ylabel("reconstruction MSE loss"); plt.title("Task B: SSL pretraining loss (MAE)")
plt.savefig(f"{CONFIG['out_dir']}/mae_ssl_loss_curve.png", dpi=150); plt.show()

# qualitative check AFTER continued pretraining -- reconstructions should look noticeably sharper/
# more food-domain-specific than the "before_ssl" plot above
visualize_mae_reconstruction(mae_model, sample_batch, n_show=4, tag="after_ssl")

# discard the pixel-reconstruction decoder -- downstream segmentation only needs the encoder
pretrained_encoder = mae_model.vit
print(f"Encoder-only params going into Task C/D: {sum(p.numel() for p in pretrained_encoder.parameters())/1e6:.1f}M")

## Task C: SegFormer‑style All‑MLP decoder on MAE encoder (adapter + full model)

In [ ]:
class SegFormerStyleHead(nn.Module):
    """Re-implementation of HF SegformerDecodeHead's All-MLP head (Part-A's best design),
    adapted for same-resolution multi-depth ViT taps (see get_multilevel_tokens adapter)."""
    def __init__(self, in_channels_list, embed_dim, num_classes, dropout=0.1):
        super().__init__()
        self.linear_c = nn.ModuleList([nn.Linear(c, embed_dim) for c in in_channels_list])
        self.linear_fuse = nn.Conv2d(embed_dim * len(in_channels_list), embed_dim, kernel_size=1, bias=False)
        self.batch_norm = nn.BatchNorm2d(embed_dim)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Conv2d(embed_dim, num_classes, kernel_size=1)

    def forward(self, feature_maps):
        projected = []
        for fmap, proj in zip(feature_maps, self.linear_c):
            B, C, H, W = fmap.shape
            x = fmap.flatten(2).transpose(1, 2)      # (B, HW, C)
            x = proj(x)                               # (B, HW, embed_dim)
            x = x.transpose(1, 2).reshape(B, -1, H, W)
            projected.append(x)                       # all already same (H,W) -> no upsample needed here
        x = torch.cat(projected, dim=1)
        x = self.linear_fuse(x)
        x = self.batch_norm(x)
        x = self.activation(x)
        x = self.dropout(x)
        return self.classifier(x)


class MAESegModel(nn.Module):
    def __init__(self, encoder, layer_idxs, patch_size, embed_dim_decoder, num_classes, freeze_encoder=False):
        super().__init__()
        self.encoder = encoder            # a ViTMAEModel (decoder already discarded, see Task B)
        self.layer_idxs = layer_idxs
        self.patch_size = patch_size
        hidden = encoder.config.hidden_size
        self.head = SegFormerStyleHead([hidden] * len(layer_idxs), embed_dim_decoder, num_classes)
        self.set_encoder_frozen(freeze_encoder)

    def set_encoder_frozen(self, freeze: bool):
        for p in self.encoder.parameters():
            p.requires_grad = not freeze
        self.frozen = freeze

    def forward(self, pixel_values):
        H, W = pixel_values.shape[-2:]
        maps = get_multilevel_tokens(self.encoder, pixel_values, self.layer_idxs, self.patch_size)
        logits = self.head(maps)
        return F.interpolate(logits, size=(H, W), mode="bilinear", align_corners=False)

## Frozen vs fine‑tuned ablation (required by PDF for ≥1 SSL method — done here for MAE)

In [ ]:
def quick_train_eval(freeze_encoder, epochs=8, tag=""):
    model = MAESegModel(copy.deepcopy(pretrained_encoder), CONFIG["layer_taps"], CONFIG["patch_size"],
                         CONFIG["decoder_embed_dim"], CONFIG["num_classes"], freeze_encoder).to(DEVICE)
    params = [{"params": model.head.parameters(), "lr": CONFIG["ft_lr_decoder"]}]
    if not freeze_encoder:
        params.append({"params": model.encoder.parameters(), "lr": CONFIG["ft_lr_encoder"]})
    opt = torch.optim.AdamW(params, weight_decay=CONFIG["ft_weight_decay"])
    ce = nn.CrossEntropyLoss()

    for ep in range(epochs):
        model.train()
        for imgs, masks in ft_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits = model(imgs)
            loss = ce(logits, masks)
            opt.zero_grad(); loss.backward(); opt.step()

    model.eval()
    inter = torch.zeros(CONFIG["num_classes"]); union = torch.zeros(CONFIG["num_classes"])
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs).argmax(1)
            for c in range(CONFIG["num_classes"]):
                p, g = (preds == c), (masks == c)
                inter[c] += (p & g).sum().item(); union[c] += (p | g).sum().item()
    miou = (inter / union.clamp(min=1)).mean().item()
    print(f"[{tag}] {epochs}-epoch probe mIoU on test split: {miou:.4f}")
    return miou

miou_frozen   = quick_train_eval(freeze_encoder=True,  tag="FROZEN (linear probe)")
miou_finetune = quick_train_eval(freeze_encoder=False, tag="FINE-TUNED")

CONFIG["encoder_mode"] = "finetune" if miou_finetune >= miou_frozen else "frozen"
print(f"-> Chosen strategy for the full Task D run: {CONFIG['encoder_mode']}  "
      f"(frozen={miou_frozen:.4f} vs finetuned={miou_finetune:.4f})")

## Task D: full supervised fine‑tuning (50 epochs, monitored on test split)

In [ ]:
freeze_final = (CONFIG["encoder_mode"] == "frozen")
model = MAESegModel(pretrained_encoder, CONFIG["layer_taps"], CONFIG["patch_size"],
                     CONFIG["decoder_embed_dim"], CONFIG["num_classes"], freeze_final).to(DEVICE)

param_groups = [{"params": model.head.parameters(), "lr": CONFIG["ft_lr_decoder"]}]
if not freeze_final:
    param_groups.append({"params": model.encoder.parameters(), "lr": CONFIG["ft_lr_encoder"]})
optimizer_ft = torch.optim.AdamW(param_groups, weight_decay=CONFIG["ft_weight_decay"])
scheduler_ft = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=CONFIG["ft_epochs"])
criterion = nn.CrossEntropyLoss()

def compute_miou(model, loader):
    model.eval()
    inter = torch.zeros(CONFIG["num_classes"]); union = torch.zeros(CONFIG["num_classes"])
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs).argmax(1)
            for c in range(CONFIG["num_classes"]):
                p, g = (preds == c), (masks == c)
                inter[c] += (p & g).sum().item(); union[c] += (p | g).sum().item()
    return (inter / union.clamp(min=1)).mean().item()

train_loss_hist, monitor_miou_hist, ft_epoch_times = [], [], []
best_miou, best_state = -1, None

for epoch in range(CONFIG["ft_epochs"]):
    t0 = time.time()
    model.train()
    running = 0.0
    for imgs, masks in ft_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        logits = model(imgs)
        loss = criterion(logits, masks)
        optimizer_ft.zero_grad(); loss.backward(); optimizer_ft.step()
        running += loss.item()
    scheduler_ft.step()
    epoch_loss = running / len(ft_loader)
    monitor_miou = compute_miou(model, test_loader)   # test split double-role: monitoring here
    epoch_time = time.time() - t0

    train_loss_hist.append(epoch_loss); monitor_miou_hist.append(monitor_miou); ft_epoch_times.append(epoch_time)
    print(f"[FT] epoch {epoch+1}/{CONFIG['ft_epochs']} | train_loss={epoch_loss:.4f} | "
          f"monitor_mIoU(test)={monitor_miou:.4f} | {epoch_time:.1f}s")

    if monitor_miou > best_miou:
        best_miou, best_state = monitor_miou, copy.deepcopy(model.state_dict())

torch.save(best_state, f"{CONFIG['out_dir']}/mae_seg_best.pth")
print(f"Avg per-epoch FT time: {np.mean(ft_epoch_times):.1f}s | Best monitor mIoU: {best_miou:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11,4))
axes[0].plot(train_loss_hist); axes[0].set_title("FT loss (labelled valid split)"); axes[0].set_xlabel("epoch")
axes[1].plot(monitor_miou_hist); axes[1].set_title("Monitoring mIoU (test split)"); axes[1].set_xlabel("epoch")
plt.tight_layout(); plt.savefig(f"{CONFIG['out_dir']}/mae_ft_curves.png", dpi=150); plt.show()

model.load_state_dict(best_state)

## Task E: final test metrics + label‑efficiency comparison

In [ ]:
model.eval()
num_c = CONFIG["num_classes"]
inter = torch.zeros(num_c); union = torch.zeros(num_c); correct_c = torch.zeros(num_c); total_c = torch.zeros(num_c)
conf_mat = torch.zeros(num_c, num_c, dtype=torch.int64)
correct_px, total_px = 0, 0
dice_inter = torch.zeros(num_c); dice_sum = torch.zeros(num_c)

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = model(imgs).argmax(1)
        correct_px += (preds == masks).sum().item(); total_px += masks.numel()
        for c in range(num_c):
            p, g = (preds == c), (masks == c)
            inter[c] += (p & g).sum().item(); union[c] += (p | g).sum().item()
            dice_inter[c] += (p & g).sum().item(); dice_sum[c] += p.sum().item() + g.sum().item()
            correct_c[c] += (p & g).sum().item(); total_c[c] += g.sum().item()

        # --- vectorized confusion matrix update (replaces the slow per-pixel Python loop) ---
        t_flat = masks.flatten().cpu()
        p_flat = preds.flatten().cpu()
        idx = t_flat * num_c + p_flat
        conf_mat += torch.bincount(idx, minlength=num_c * num_c).reshape(num_c, num_c)

per_class_iou = (inter / union.clamp(min=1)).numpy()
per_class_dice = (2 * dice_inter / dice_sum.clamp(min=1)).numpy()
per_class_acc = (correct_c / total_c.clamp(min=1)).numpy()

test_mIoU = float(np.mean(per_class_iou))
pixel_acc = correct_px / total_px
mean_pixel_acc = float(np.mean(per_class_acc))

mae_summary = {
    "model": f"MAE-Seg ({CONFIG['hf_backbone']}, mask_ratio={CONFIG['mask_ratio']}, {CONFIG['encoder_mode']})",
    "test_mIoU": test_mIoU,
    "test_pixel_acc": pixel_acc,
    "test_mean_pixel_acc": mean_pixel_acc,
    "labelled_finetune_images": len(valid_imgs),
    "num_params_millions": sum(p.numel() for p in model.parameters()) / 1e6,
}
print(json.dumps(mae_summary, indent=2))

per_class_df = pd.DataFrame({
    "class_id": range(num_c), "class_name": CLASS_NAMES,
    "IoU": per_class_iou, "PixelAcc": per_class_acc, "Dice": per_class_dice,
})
per_class_df.to_csv(f"{CONFIG['out_dir']}/mae_test_metrics_per_class.csv", index=False)

# save confusion matrix too (useful for Task F / report figure)
conf_df = pd.DataFrame(conf_mat.numpy(), index=CLASS_NAMES, columns=CLASS_NAMES)
conf_df.to_csv(f"{CONFIG['out_dir']}/mae_confusion_matrix.csv")

# --- label-efficiency comparison vs Part-A baseline (and, if available, vs the DINOv2 SSL run) ---
PART_A_BASELINE = {"model": "SegFormer-B0 (Part-A, fully supervised)",
                    "test_mIoU": 0.4800178820641928, "train_images": len(train_imgs)}

comparison_rows = [
    {"Model": PART_A_BASELINE["model"], "Test mIoU": PART_A_BASELINE["test_mIoU"],
     "Labelled images used": PART_A_BASELINE["train_images"]},
    {"Model": mae_summary["model"], "Test mIoU": mae_summary["test_mIoU"],
     "Labelled images used": len(valid_imgs)},
]

# optional: pull in a DINOv2 run's results.json from a sibling notebook if one is mounted/available
_dinov2_results_candidates = ["/kaggle/working/dinov2_outputs/results.json",
                               "/kaggle/input/dinov2-outputs/results.json"]
for _p in _dinov2_results_candidates:
    if os.path.exists(_p):
        _other = json.load(open(_p))
        if "dinov2" in _other:
            comparison_rows.append({"Model": _other["dinov2"]["model"],
                                     "Test mIoU": _other["dinov2"]["test_mIoU"],
                                     "Labelled images used": _other["dinov2"]["labelled_finetune_images"]})
        break

comparison = pd.DataFrame(comparison_rows)
comparison["Label ratio vs Part-A"] = comparison["Labelled images used"] / PART_A_BASELINE["train_images"]
print(comparison.to_string(index=False))

## Task F: error analysis (worst‑K predictions)

In [ ]:
def per_image_miou(model, loader, k=8):
    model.eval()
    scores = []
    with torch.no_grad():
        for bi, (imgs, masks) in enumerate(loader):
            imgs_d, masks_d = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs_d).argmax(1)
            for i in range(imgs.shape[0]):
                p, g = preds[i], masks_d[i]
                ious = []
                for c in torch.unique(g).tolist():
                    pc, gc = (p == c), (g == c)
                    u = (pc | gc).sum().item()
                    if u > 0:
                        ious.append((pc & gc).sum().item() / u)
                scores.append((np.mean(ious), bi, i, imgs[i].cpu(), masks[i].cpu(), p.cpu()))
    scores.sort(key=lambda x: x[0])
    return scores[:k]

worst = per_image_miou(model, test_loader, k=8)
fig, axes = plt.subplots(len(worst), 3, figsize=(9, 3*len(worst)))
for row, (score, bi, i, img, gt, pred) in enumerate(worst):
    axes[row,0].imshow(denorm(img)); axes[row,0].set_title(f"image (IoU={score:.3f})"); axes[row,0].axis("off")
    axes[row,1].imshow(gt, cmap="tab20"); axes[row,1].set_title("ground truth"); axes[row,1].axis("off")
    axes[row,2].imshow(pred, cmap="tab20"); axes[row,2].set_title("prediction"); axes[row,2].axis("off")
plt.tight_layout(); plt.savefig(f"{CONFIG['out_dir']}/mae_worst_predictions.png", dpi=150); plt.show()

In [ ]:
import seaborn as sns

# --- normalize confusion matrix by row (true class) so each row sums to 1 ---
conf_np = conf_mat.numpy().astype(np.float64)
row_sums = conf_np.sum(axis=1, keepdims=True)
conf_norm = np.divide(conf_np, row_sums, out=np.zeros_like(conf_np), where=row_sums != 0)

# --- full heatmap (68x68 -- readable at this size, but small font) ---
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(conf_norm, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cmap="viridis", vmin=0, vmax=1, ax=ax, cbar_kws={"label": "fraction of true class"})
ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
ax.set_title("MAE-Seg — Normalized Confusion Matrix (test split)")
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.tight_layout()
plt.savefig(f"{CONFIG['out_dir']}/mae_confusion_matrix_heatmap.png", dpi=200)
plt.show()

# --- top-20 off-diagonal confusions (the pairs actually worth discussing in Task F) ---
off_diag = []
for i in range(num_c):
    for j in range(num_c):
        if i != j and conf_mat[i, j] > 0:
            off_diag.append((CLASS_NAMES[i], CLASS_NAMES[j], int(conf_mat[i, j]), conf_norm[i, j]))

confusion_df = pd.DataFrame(off_diag, columns=["true_class", "predicted_as", "pixel_count", "fraction_of_true_class"])
confusion_df = confusion_df.sort_values("fraction_of_true_class", ascending=False).head(20)
confusion_df.to_csv(f"{CONFIG['out_dir']}/mae_top_confusions.csv", index=False)
print(confusion_df.to_string(index=False))

## Append to shared results.json (for the Final Comparison notebook)

results_path = f"{CONFIG['out_dir']}/results.json"
all_results = {}
if os.path.exists(results_path):
    all_results = json.load(open(results_path))
all_results["mae"] = {**mae_summary,
                       "frozen_probe_miou": miou_frozen, "finetuned_probe_miou": miou_finetune,
                       "chosen_strategy": CONFIG["encoder_mode"],
                       "mask_ratio": CONFIG["mask_ratio"],
                       "mae_avg_epoch_time_s": float(np.mean(mae_epoch_times)),
                       "ft_avg_epoch_time_s": float(np.mean(ft_epoch_times))}
json.dump(all_results, open(results_path, "w"), indent=2)
print("Saved:", results_path)